# Retranslation

En esta sección se tiene la retraducción de las premisas FOL -> Lenguaje Natural. Se evalúa usando LogicSim+

In [1]:
import pandas as pd
import re, spacy

### Manual Filtering (NO LES ENCANTA FILTRAR LO QUE DICE UN LLM????)

In [2]:
def get_ret(path, test):
    if test:
        mid = 452
        path = path.format('test')
    else:
        mid = 406
        path = path.format('validation')
    dataset = pd.read_csv(path)
    dataset = dataset.drop(columns = ["Unnamed: 0"])
    infer = dataset[mid:]
    infer = infer.reset_index()
    infer = infer.drop(columns=["index"])
    return infer

test = get_ret('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv', True)
val = get_ret('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv', False)
fol_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines =True)
fol_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines =True)

In [3]:
dataset_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-Distill-Qwen-7B_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/gemma-3-4b-it_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/gemma-3-12b-it_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/gpt-oss-20b_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-4B-FP8_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-8B-FP8_full.csv',  
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-14B-FP8_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-4B_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-9B_full.csv'   
]

In [4]:
def get_nice_dict(regex, instance, const):
	"""
		A partir de una regex, extrae todos los valores encontrados y los formatea para tener un diccionario con apariciones.

		regex = str ; Expresión regular a buscar.
		const = bool ; Determina si estamos analizando constantes. Esto permite que se realice un filtro extra en el código.
	"""
	lista = []
	for _ in instance:
		aux = re.finditer(regex, _)
		for expression in aux:
			if const:
				regex_list = expression.group()[1:-1]
				regex_list = regex_list.split(',')
				for j in regex_list:
					if len(j) > 1:
						constant = re.sub(' ', '', j)
						lista.append(constant)
			else:
				lista.append(expression.group())

	set_set = list(set(lista))
	dict_dict = {}
	for _ in set_set:
		dict_dict["{}".format(_)] = lista.count(_)

	return dict_dict

In [5]:
# Si dios quiere este es un filtrado sencillo. POR FAVOR.
def ez_filter(path, val):
    if val:
        folio = fol_val
        path = path.format('validation')
    else:
        folio = fol_test
        path = path.format('test')

    dataset = get_ret(path, not val)

    filtered_list = []
    for i in range(len(dataset)):
        first_split = dataset["Full"][i].split('------')[0]
        second_split = first_split.split('\n')[0]
        #print(folio['conclusion-FOL'][i])
        #print(second_split)
        #print('-' * 30)
        filtered_list.append(second_split)
    
    return filtered_list

In [ ]:
#for i in range(len(dataset_path)):
#    checkpoint_name = dataset_path[i].split('/')[-1][:-9]
#    print('='*10, checkpoint_name, '='*10)
#    test_filter = ez_filter(dataset_path[i], True)
#    val_filter = ez_filter(dataset_path[i], False)

#    test_dict = {"Retranslation": test_filter}
#    val_dict = {"Retranslation": val_filter}

#    test_df = pd.DataFrame(data = test_dict)
#    val_df = pd.DataFrame(data = val_dict)

#    test_df.to_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/filtered/retranslation/RET_{}.csv'.format(checkpoint_name))
#    val_df.to_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/filtered/retranslation/RET_{}.csv'.format(checkpoint_name))

========== DeepSeek-R1-0528-Qwen3-8B ==========
========== DeepSeek-R1-Distill-Qwen-7B ==========
========== gemma-3-4b-it ==========
========== gemma-3-12b-it ==========
========== gpt-oss-20b ==========
========== Qwen3-4B-FP8 ==========
========== Qwen3-8B-FP8 ==========
========== Qwen3-14B-FP8 ==========
========== Qwen3.5-4B ==========
========== Qwen3.5-9B ==========


In [6]:
def compare_logops(texto, fol_texto):
    """
    Compara cuantificadores FOL vs NL.

    A nivel de código esto es asqueroso cabrón.
    
    Idea:
    LogOps(FOLIO) - LogOps(NL) = Una parte de LogicSim+Ret
    Lo idea es que sea un valor cercano a 0. Si es mayor que 0 entonces el modelo puede estar "optimizando" la
    traducción. Pero si el valor es negativo implica que está agregando cosas que no debería de estar diciendo.
    """
    forall = fol_texto.count('∀')
    land = fol_texto.count('∧')
    implies = fol_texto.count('→')
    xor = fol_texto.count('⊕')
    neg = fol_texto.count('¬')
    lor = fol_texto.count('∨')
    exists = fol_texto.count('∃')
    #print("-"*5, 'Cuantificadores', '-'*5)
    #print("∀: {}. ∧: {}. →: {}. ⊕: {}. ¬: {}. ∨: {}. ∃: {}.".format(forall, land, implies, xor, neg, lor, exists)) 

    texto = texto.lower()
    every, aand, then, either, nott, orr, exist = 0, 0, 0, 0, 0, 0, 0
    every = texto.count('every')
    aand = texto.count('and')
    then = texto.count('then')
    either = texto.count('either')
    nott = texto.count('not')
    orr = texto.count('or')
    exist = texto.count('exists')
    #print("Every: {}. And: {}. Then: {}. Either: {}. Not: {}. Or: {}. Exists: {}.".format(every, aand, then, either, nott, orr, exist)) 
    total_fol = forall + land + implies + xor + neg + lor + exists
    total_nl = every + aand + then + either + nott + orr + exist
    logops_diffs = abs(total_fol - total_nl)

    print("Diferencias totales: {}".format(logops_diffs))


def predicates_nouns(texto, fol_texto):
    # Etiquetamos POST y nos fijamos en los Nouns
    # Extraemos las variables y las comparamos con los Nouns.
    const_dict = get_nice_dict(r'(\([A-z]+(\, [A-z]+){0,}\))', fol_texto, True)
    
    nlp = spacy.load("en_core_web_lg")
    doc = nlp(texto)
    #for token in doc:
    #    print(token.text, token.pos_)
    tagged_nl = [[token.text, token.pos_] for token in doc]



# 2. Asociamos variables FOL <-> NL
# 3. Asociamos predicados FOL <-> NL

In [7]:
owo = pd.read_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/filtered/retranslation/RET_DeepSeek-R1-0528-Qwen3-8B.csv')
owo = owo.drop(columns = ['Unnamed: 0'])
test = owo['Retranslation'][0]

In [14]:
nlp = spacy.load('en_core_web_lg')
doc = nlp(test)
tagged = [[token.text, token.pos_] for token in doc]
nouns = [elem[0] for elem in tagged if elem[1] in ['PROPN', 'NOUN']]
nouns

['Stephen', 'Curry', 'athlete']

In [ ]:
aux = {
    'hola(aaa)': 3,
    'hola(bbb)': 5,
    'kkdecola(easa)': 5
}

ax = list(aux.keys())

aux2 = {
    'hola':3,
    'hola':5, 
    'kkdecola':5
}

{'hola', 'kkdecola'}

In [ ]:
# Contamos apariciones individuales de cada predicado.
for i in range(10):
    const_dict = get_nice_dict(r'(\([A-z]+(\, [A-z]+){0,}\))', [fol_test['conclusion-FOL'][i]], True)
    pred_dict = get_nice_dict(r'[A-z]+\(([A-z]+(,? [A-z]+)*)\)', [fol_test['conclusion-FOL'][i]], False)

    print(pred_dict)
    key = list(pred_dict.keys())
    if len(key) > 0:
        for j in range(len(pred_dict)):
            split = key[j].split('(')[0]
            if split in list(pred_dict.keys()):
                pred_dict[split] += pred_dict.pop(key[j])
            else:
                pred_dict[split] = pred_dict.pop(key[j])
        print(pred_dict)
    else:
        print('No hay predicados')

    print('----')
    #pred_lists = list(pred_dict.keys())
    #pred_lists = [elem.split(r'\(')[0] for elem in pred_lists]

    #print(const_dict)
    #print(pred_dict)
    #print('----')

{'Athlete(stephenCurry)': 1}
{'Athlete': 1}
----
{'CenterBack(stephenCurry)': 1, 'Athlete(stephenCurry)': 1, 'Professional(stephenCurry)': 1}
{'CenterBack': 1, 'Athlete': 1, 'Professional': 1}
----
{'CenterBack(stephenCurry)': 1, 'Athlete(stephenCurry)': 1, 'Professional(stephenCurry)': 1}
{'CenterBack': 1, 'Athlete': 1, 'Professional': 1}
----
{'JoinIn(tyler, studentGovernment, highSchool)': 1}
{'JoinIn': 1}
----
{'ManyOpportunityToImprove(tyler, communicationSkill)': 1, 'StayAloneIn(tyler, highSchool)': 2, 'EngagedWithIn(x, community)': 1, 'adjustedWithin(x, community)': 1}
{'ManyOpportunityToImprove': 1, 'StayAloneIn': 2, 'EngagedWithIn': 1, 'adjustedWithin': 1}
----
{'MajorIn(tim, computerScience)': 1}
{'MajorIn': 1}
----
{'From(tim, yaleRunningClub)': 1, 'MajorIn(tim, computerScience)': 1}
{'From': 1, 'MajorIn': 1}
----
{}
No hay predicados
----
{'Noisy(x)': 1, 'Rose(x)': 1, 'Noisy(y)': 1, 'Rose(y)': 1}
{'Noisy': 2, 'Rose': 2}
----
{'Study(murphy, lightDetection)': 1, 'Study(murph

In [29]:
for i in range(len(test['Full'])):
    compare_logops(a[i], fol_test['conclusion-FOL'][i])

----- Cuantificadores -----
∀: 0. ∧: 0. →: 0. ⊕: 0. ¬: 0. ∨: 0. ∃: 0.
Every: 0. And: 0. Then: 0. Either: 0. Not: 0. Or: 0. Exists: 0.
Diferencias totales: 0
----- Cuantificadores -----
∀: 0. ∧: 2. →: 0. ⊕: 0. ¬: 0. ∨: 0. ∃: 0.
Every: 0. And: 2. Then: 0. Either: 0. Not: 0. Or: 0. Exists: 0.
Diferencias totales: 0
----- Cuantificadores -----
∀: 0. ∧: 1. →: 1. ⊕: 0. ¬: 1. ∨: 0. ∃: 0.
Every: 0. And: 1. Then: 1. Either: 0. Not: 1. Or: 0. Exists: 0.
Diferencias totales: 0
----- Cuantificadores -----
∀: 0. ∧: 0. →: 0. ⊕: 0. ¬: 0. ∨: 0. ∃: 0.
Every: 0. And: 0. Then: 0. Either: 0. Not: 0. Or: 0. Exists: 0.
Diferencias totales: 0
----- Cuantificadores -----
∀: 0. ∧: 1. →: 1. ⊕: 2. ¬: 2. ∨: 0. ∃: 0.
Every: 0. And: 1. Then: 1. Either: 2. Not: 2. Or: 4. Exists: 0.
Diferencias totales: 4
----- Cuantificadores -----
∀: 0. ∧: 0. →: 0. ⊕: 0. ¬: 0. ∨: 0. ∃: 0.
Every: 0. And: 0. Then: 0. Either: 0. Not: 0. Or: 1. Exists: 0.
Diferencias totales: 1
----- Cuantificadores -----
∀: 0. ∧: 1. →: 0. ⊕: 0. ¬: 2. 